# Hanabi - Data Engineering

The data is obtained from https://github.com/yawgmoth/HanabiData.

In [1]:
import re
import random
import os
import pandas as pd

The repo is 9 years old and contains code made with Python 2. They included the following code to create the deck with a random seed given in the game logs:

In [2]:
# def make_deck(seed):
#     random.seed(seed)
#     deck = []
#     for col in ["green", "yellow", "white", "blue", "red"]:
#         for num, cnt in enumerate([3,2,2,2,1]):
#             for i in xrange(cnt):
#                 deck.append((col, num+1))
#     random.shuffle(deck)
#     return deck

We had to change the logic to Python 3 and used GenAI to create a function which simulates the Python 2 shuffle:

In [3]:
def py2_shuffle(items):
    for i in range(len(items) - 1, 0, -1):
        j = int(random.random() * (i + 1))
        items[i], items[j] = items[j], items[i]

def make_deck(seed):
    colors = ["green", "yellow", "white", "blue", "red"]
    random.seed(seed)
    deck = []
    for col in colors:
        for num, cnt in enumerate([3, 2, 2, 2, 1]):
            for i in range(cnt):
                deck.append((col, num + 1))
    
    py2_shuffle(deck)
    return deck

This function extracts the start cards for both players by using the make_deck function:

In [4]:
def process_hanabi_log(log_content, filename):
    lines = log_content.split('\n')
    
    seed = None
    final_score = None
    first_player = None

    for line in lines:
        line = line.strip()
        if line.startswith("Treatment:"): #after the treatment, we can find the seed for the deck
            match = re.search(r",\s*(\d+)\)", line)
            if match:
                seed = int(match.group(1)) #the seed used for the make_deck function
        
        elif line.startswith("MOVE:") and first_player is None: #defines the first player
            first_player = int(line.split()[1]) #0 is the AI, 1 is the human player
            
        elif line.startswith("Score"): 
            final_score = int(line.split()[-1]) #the final score is found at the end of the line after "Score:"

    deck = make_deck(seed)
    
    if first_player == 0: #0 is the AI
        p1_cards, p2_cards = deck[0:5], deck[5:10] # p1 is the AI
    else: #1 is the human player
        p2_cards, p1_cards = deck[0:5], deck[5:10] # p2 is the human player

  #one hot encoding
    colors = ["green", "yellow", "white", "blue", "red"]
    row_data = {"file_source": filename, "final_score": final_score}
    
    for player_prefix, hand in [("p1", p1_cards), ("p2", p2_cards)]:
        for col in colors:
            for num in range(1, 6):
                col_name = f"{player_prefix}_{col}{num}"
                row_data[col_name] = hand.count((col, num)) #counts how many times the card appears
                
    return row_data

In [5]:
folder_path = 'log' 
all_games_data = []

filenames = [f for f in os.listdir(folder_path) if f.startswith("game") and f.endswith(".log")]

for filename in filenames:
    file_path = os.path.join(folder_path, filename)
    with open(file_path, 'r', encoding='utf-8') as file:
        log_content = file.read()
        game_row = process_hanabi_log(log_content, filename)
        if game_row:
            all_games_data.append(game_row)

df = pd.DataFrame(all_games_data)

In [6]:
df.head()

,file_source,final_score,p1_green1,p1_green2,p1_green3,p1_green4,p1_green5,p1_yellow1,p1_yellow2,p1_yellow3,...,p2_blue1,p2_blue2,p2_blue3,p2_blue4,p2_blue5,p2_red1,p2_red2,p2_red3,p2_red4,p2_red5
0,game003d9bcb9d27dacf.log,15,1,0,0,0,0,1,1,0,...,0,0,0,0,0,1,0,0,1,0
1,game0073425f0b25520f.log,17,1,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,game007c0324e3f7b88a.log,19,0,0,0,2,0,1,0,0,...,0,1,0,0,0,0,0,1,0,0
3,game007e20cb959b48fa.log,12,1,1,0,0,0,0,0,0,...,2,0,0,1,0,0,0,0,0,0
4,game00abb0a354af727f.log,9,0,1,1,0,0,0,0,0,...,1,1,0,0,0,1,0,0,0,0


In [7]:
df.to_csv("hanabi_dataset.csv", index=False)

Example validation check:

In [8]:
log_name = 'game00abb0a354af727f.log'
row = df[df['file_source'] == log_name].iloc[0]

p1_start = [c.replace('p1_', '') for c in row.index if c.startswith('p1_') and row[c] == 1]
p2_start = [c.replace('p2_', '') for c in row.index if c.startswith('p2_') and row[c] == 1]

print(row['final_score'])
print(p1_start)
print(p2_start)

9
['green2', 'green3', 'white2', 'white4', 'red5']
['green2', 'green4', 'blue1', 'blue2', 'red1']


Game log:

    MOVE: 0 1 None 1 None 1
    intentional hints You about all their 1 hints remaining: 7
    You has green 4, blue 1, blue 2, green 2, red 1
    MOVE: 1 0 None 0 0 None
    You hints intentional about all their green cards hints remaining: 6
    intentional has white 2, green 3, white 4, green 2, red 5

-> works